# Bayesian Hyperparameter Optimisation — GAE Bubble Detection

Three sequential rounds using [Optuna](https://optuna.org/) TPE sampler:
- **Round 1 (ACTIVE):** Training dynamics — `lr`, `weight_decay`, `K` (10–44), `patience`
- **Round 2 (COMMENTED):** Architecture — `hidden_dim`, `encoder_channels`, `num_diffusion_steps`, `dropout`, `gat_heads`, `embedding_dim`, `gat_output_dim`
- **Round 3 (COMMENTED):** Graph structure — `k_neighbors`, `corr_threshold`

**Objective:** maximise validation AUC averaged over FW=22/CT=-0.20 and FW=22/CT=-0.30.

**Data splits:** Train 2012-2016 | Val 2017-mid2019 | Test mid2019-2024 (held out).

In [52]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler
import warnings, os, traceback
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial noise
from tqdm import tqdm
from torch.optim import Adam
import matplotlib.pyplot as plt


In [53]:

# works out correlation matrix for returns in window size K ending at t - so up to time t
def correlation_matrix(returns, t, K, eps=0.0, active=None):
    t = pd.to_datetime(t)
    # 1. Slice the window
    window = returns.loc[t - pd.Timedelta(days=K*1.5): t]
    # 2. Filter for active and non-zero stocks
    if active is not None:
        window = window[active]
    if eps == 0.0:
        window = window.loc[:, (window.fillna(0.0) != 0.0).any(axis=0)]
    else:
        window = window.loc[:, (window.fillna(0.0).abs() > eps).any(axis=0)]
        
    active_cols = window.columns.tolist()
    # 3. Vectorized Math (NumPy is faster here)
    X = window.values # Shape: (Time, Stocks)
    X = np.nan_to_num(X) # Handle any remaining NaNs
    
    centered = X - np.mean(X, axis=0)
    # matrix multiplication: (Stocks, Time) @ (Time, Stocks) -> (Stocks, Stocks)
    cov = (centered.T @ centered) / (X.shape[0] - 1)
    
    std = np.sqrt(np.diag(cov))
    # Outer product of standard deviations with epsilon for stability
    denominator = np.outer(std, std) + 1e-9
    corr = cov / denominator
    
    return np.clip(corr, -1.0, 1.0), active_cols # Clip to ensure valid corr range

    # corr_matrix = window.corr().values
    # corr_matrix = np.nan_to_num(corr_matrix, nan=0.0, posinf=0.0, neginf=0.0)

    # return corr_matrix, active_cols

# function takes in historical returns for each t with window size K 
# and computes initial node embeddings as H = US where Y = U S V^T is the SVD of the returns matrix Y

def compute_initial_node_embeddings(returns, t, K, eps=0.0, active=None):

    
    node_embeddings = {}
    t = pd.to_datetime(t)
    windowed_returns = returns.loc[t - pd.Timedelta(days=K*1.5): t].dropna(how='all')
    
    # Check if window is empty
    if windowed_returns.empty:
        return {}, []

    window = windowed_returns.dropna(axis=1, how="all")
    if active is not None:
        # Intersect active with available columns
        valid_active = [c for c in active if c in window.columns]
        if not valid_active:
            return {}, []
        window = window[valid_active]
        
    if window.shape[0] < 2 or window.shape[1] == 0: # Need at least some data
        return {}, []

    if eps == 0.0:
        window = window.loc[:, ~(window.fillna(0.0) == 0.0).all(axis=0)]
    else:
        window = window.loc[:, ~(window.fillna(0.0).abs() <= eps).all(axis=0)]

    active_cols = window.columns.tolist()
    if not active_cols:
        return {}, []
    data_to_svd = window.fillna(0.0).values
    data_to_svd = np.nan_to_num(data_to_svd, nan=0.0, posinf=0.0, neginf=0.0)
    U, S, Vt = np.linalg.svd(data_to_svd, full_matrices=False)
    V = Vt.T
    H = V @ np.diag(S)
    H = H[:, :10]

    # Handle case where SVD returns fewer than 10 components
    if H.shape[1] < 10:
        padding = np.zeros((H.shape[0], 10 - H.shape[1]))
        H = np.hstack([H, padding])
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    H = scaler.fit_transform(H)
    H = np.nan_to_num(H, nan=0.0)  # add this

    for i, stock in enumerate(active_cols):
        node_embeddings[stock] = H[i, :] # contains only one embedding per stock at time t, embedding shape (10,)
        
    # here - store the array directly in the list to match our structure
    for stock in node_embeddings:
        node_embeddings[stock] = np.array(node_embeddings[stock])
        
    return node_embeddings, active_cols

In [54]:
# layer gives me attention weights for edges - gives A_+ and A_-
class GATLayer(torch.nn.Module):
    
    src_nodes_dim = 0  # position of source nodes in edge index
    trg_nodes_dim = 1  # position of target nodes in edge index

    nodes_dim = 0      # node dimension (the position of "N" in tensor)
    head_dim = 2       # attention head dim

    def __init__(self, num_in_features, num_out_features, num_of_heads, concat=True, activation=nn.ELU(),
                 dropout_prob=0.1, add_skip_connection=True, bias=True, log_attention_weights=False):

        super().__init__()

        self.num_of_heads = num_of_heads # number of attention heads - mine has a positive and negative head so 2
        self.num_out_features = num_out_features
        self.concat = concat  # whether we should concatenate or average the attention heads
        self.add_skip_connection = add_skip_connection

        # treat this one matrix as num_of_heads independent W matrices
        self.linear_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)

        # After we concatenate target node (node i) and source node (node j) we apply the additive scoring function
        # which gives us un-normalized score "e". we split the "a" vector - but the semantics remain the same.
        # Basically instead of doing [x, y] (concatenation, x/y are node feature vectors) and dot product with "a"
        # we instead do a dot product between x and "a_left" and y and "a_right" and we sum them up
        # these params will be updated during training (below)
        self.scoring_fn_target = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))
        self.scoring_fn_source = nn.Parameter(torch.Tensor(1, num_of_heads, num_out_features))

        # Bias is definitely not crucial to GAT - feel free to experiment (I pinged the main author, Petar, on this one)
        if bias and concat:
            self.bias = nn.Parameter(torch.Tensor(num_of_heads * num_out_features))
        elif bias and not concat:
            self.bias = nn.Parameter(torch.Tensor(num_out_features))
        else:
            self.register_parameter('bias', None) # no bias

        if add_skip_connection:
            self.skip_proj = nn.Linear(num_in_features, num_of_heads * num_out_features, bias=False)
        else:
            self.register_parameter('skip_proj', None)


        self.leakyReLU = nn.LeakyReLU(0.2)  # using 0.2 as in the paper, no need to expose every setting
        self.activation = activation
        # Probably not the nicest design but I use the same module in 3 locations, before/after features projection
        # and for attention coefficients. Functionality-wise it's the same as using independent modules.
        self.dropout = nn.Dropout(p=dropout_prob)

        self.log_attention_weights = log_attention_weights  # whether we should log the attention weights
        self.attention_weights = None  # for later visualization purposes, I cache the weights here

        self.init_params()
        
    def forward(self, data):
        # Step 1: Linear Projection + regularization

        in_nodes_features, edge_index, corr = data  # unpack data - corr contains N x N correlation matrix at time t 
        num_of_nodes = in_nodes_features.shape[self.nodes_dim]
        assert edge_index.shape[0] == 2, f'Expected edge index with shape=(2,E) got {edge_index.shape}'

        # shape = (N, FIN) where N - number of nodes in the graph, FIN - number of input features per node
        # apply the dropout to all of the input node features (as mentioned in the paper)
        # can try without this part as an experiment
        in_nodes_features = self.dropout(in_nodes_features)

        # shape = (N, FIN) * (FIN, NH*FOUT) -> (N, NH, FOUT) where NH - number of heads, FOUT - num of output features
        # We project the input node features into NH independent output features (one for each attention head)
        nodes_features_proj = self.linear_proj(in_nodes_features).view(-1, self.num_of_heads, self.num_out_features)

        # can try without this
        nodes_features_proj = self.dropout(nodes_features_proj)  # in the official GAT imp they did dropout here as well

        # Step 2:edge attention calculation

        # Apply the scoring function (* represents element-wise (a.k.a. Hadamard) product)
        # shape = (N, NH, FOUT) * (1, NH, FOUT) -> (N, NH, 1) -> (N, NH) because sum squeezes the last dimension
        # Optimization note: torch.sum() is as performant as .sum() in my experiments
        scores_source = (nodes_features_proj * self.scoring_fn_source).sum(dim=-1)
        scores_target = (nodes_features_proj * self.scoring_fn_target).sum(dim=-1)

        # We simply copy (lift) the scores for source/target nodes based on the edge index. Instead of preparing all
        # the possible combinations of scores we just prepare those that will actually be used and those are defined
        # by the edge index.
        # scores shape = (E, NH), nodes_features_proj_lifted shape = (E, NH, FOUT), E - number of edges in the graph
        scores_source_lifted, scores_target_lifted, nodes_features_proj_lifted = self.lift(scores_source, scores_target, nodes_features_proj, edge_index)
        scores_per_edge = self.leakyReLU(scores_source_lifted + scores_target_lifted)
        #scores_per_edge = self.mask_function(scores_per_edge, edge_index)
        
        src = edge_index[self.src_nodes_dim]
        trg = edge_index[self.trg_nodes_dim]

        # corr is N x N, extract per-edge correlations (E,)
        corr_e = corr[src, trg] 
        # masks (E, 1) so they broadcast across heads
        mask_pos = (corr_e >= 0).float().unsqueeze(-1)
        mask_neg = (corr_e < 0).float().unsqueeze(-1)

        # two masked attentions, each normalized over the same target-neighborhood
        att_pos = self.neighborhood_aware_softmax(scores_per_edge, trg, num_of_nodes, mask_pos)
        att_neg = self.neighborhood_aware_softmax(scores_per_edge, trg, num_of_nodes, mask_neg)

        # treat them as two separate sign heads
        attentions_per_edge = torch.cat([att_pos, att_neg], dim=1)  # (E, 2*NH, 1)
        attentions_per_edge = self.dropout(attentions_per_edge)

        return attentions_per_edge
        
    def neighborhood_aware_softmax(self, scores_per_edge, trg_index, num_of_nodes, mask):
        # Calculate the numerator. Make logits <= 0 so that e^logit <= 1 (this will improve the numerical stability)
        scores_per_edge = scores_per_edge - scores_per_edge.max()
        exp_scores_per_edge = scores_per_edge.exp()  # softmax
        exp_scores_per_edge = exp_scores_per_edge * mask

        # Calculate the denominator. shape = (E, NH)
        neigborhood_aware_denominator = self.sum_edge_scores_neighborhood_aware(exp_scores_per_edge, trg_index, num_of_nodes)
        # 1e-16 is theoretically not needed but is only there for numerical stability (avoid div by 0) - due to the
        # possibility of the computer rounding a very small number all the way to 0.
        attentions_per_edge = exp_scores_per_edge / (neigborhood_aware_denominator + 1e-16)
        # shape = (E, NH) -> (E, NH, 1) so that we can do element-wise multiplication with projected node features
        return attentions_per_edge.unsqueeze(-1)

    def sum_edge_scores_neighborhood_aware(self, exp_scores_per_edge, trg_index, num_of_nodes):
        # The shape must be the same as in exp_scores_per_edge (required by scatter_add_) i.e. from E -> (E, NH)
        trg_index_broadcasted = self.explicit_broadcast(trg_index, exp_scores_per_edge)
        # shape = (N, NH), where N is the number of nodes and NH the number of attention heads
        size = list(exp_scores_per_edge.shape)  # convert to list otherwise assignment is not possible
        size[self.nodes_dim] = num_of_nodes
        neighborhood_sums = torch.zeros(size, dtype=exp_scores_per_edge.dtype, device=exp_scores_per_edge.device)
        # position i will contain a sum of exp scores of all the nodes that point to the node i (as dictated by the
        # target index)
        neighborhood_sums.scatter_add_(self.nodes_dim, trg_index_broadcasted, exp_scores_per_edge)
        # Expand again so that we can use it as a softmax denominator. e.g. node i's sum will be copied to
        # all the locations where the source nodes pointed to i (as dictated by the target index)
        # shape = (N, NH) -> (E, NH)
        return neighborhood_sums.index_select(self.nodes_dim, trg_index)

    def aggregate_neighbors(self, nodes_features_proj_lifted_weighted, edge_index, in_nodes_features, num_of_nodes):
        size = list(nodes_features_proj_lifted_weighted.shape)  # convert to list otherwise assignment is not possible
        size[self.nodes_dim] = num_of_nodes  # shape = (N, NH, FOUT)
        out_nodes_features = torch.zeros(size, dtype=in_nodes_features.dtype, device=in_nodes_features.device)

        # shape = (E) -> (E, NH, FOUT)
        trg_index_broadcasted = self.explicit_broadcast(edge_index[self.trg_nodes_dim], nodes_features_proj_lifted_weighted)
        # aggregation step - we accumulate projected, weighted node features for all the attention heads
        # shape = (E, NH, FOUT) -> (N, NH, FOUT)
        out_nodes_features.scatter_add_(self.nodes_dim, trg_index_broadcasted, nodes_features_proj_lifted_weighted)

        return out_nodes_features

    def lift(self, scores_source, scores_target, nodes_features_matrix_proj, edge_index):
        #Lift duplicates certain vectors depending on the edge index.
        #One of the tensor dims goes from N -> E (that's where the "lift" comes from)

        src_nodes_index = edge_index[self.src_nodes_dim]
        trg_nodes_index = edge_index[self.trg_nodes_dim]

        # Using index_select is faster than "normal" indexing (scores_source[src_nodes_index]) in PyTorch!
        scores_source = scores_source.index_select(self.nodes_dim, src_nodes_index)
        scores_target = scores_target.index_select(self.nodes_dim, trg_nodes_index)
        nodes_features_matrix_proj_lifted = nodes_features_matrix_proj.index_select(self.nodes_dim, src_nodes_index)

        return scores_source, scores_target, nodes_features_matrix_proj_lifted

    def explicit_broadcast(self, this, other):
        # Append singleton dimensions until this.dim() == other.dim()
        for _ in range(this.dim(), other.dim()):
            this = this.unsqueeze(-1)

        # Explicitly expand so that shapes are the same
        return this.expand_as(other)

    def init_params(self): # TODO: try changing initialization
        """
        The reason we're using Glorot (aka Xavier uniform) initialization is because it's a default TF initialization:
            https://stackoverflow.com/questions/37350131/what-is-the-default-variable-initializer-in-tensorflow

        The original repo was developed in TensorFlow (TF) and they used the default initialization.
        Feel free to experiment - there may be better initializations depending on your problem.

        """
        nn.init.xavier_uniform_(self.linear_proj.weight)
        nn.init.xavier_uniform_(self.scoring_fn_target)
        nn.init.xavier_uniform_(self.scoring_fn_source)

        if self.bias is not None:
            torch.nn.init.zeros_(self.bias)

In [55]:
from torch.optim import Adam

class GAT(torch.nn.Module): # gives me attention weights matrices for edges - gives A_+ and A_-

    def __init__(self, num_of_layers, num_heads_per_layer, num_features_per_layer, add_skip_connection=True, bias=True,
                 dropout=0.1, log_attention_weights=False):
        super().__init__()
        assert num_of_layers == len(num_heads_per_layer) == len(num_features_per_layer) - 1, f'Enter valid arch params.'

        num_heads_per_layer = [1] + num_heads_per_layer  # trick - so that I can nicely create GAT layers below

        gat_layers = []  # collect GAT layers
        for i in range(num_of_layers):
            layer = GATLayer(
                num_in_features=num_features_per_layer[i] * num_heads_per_layer[i],  # consequence of concatenation
                num_out_features=num_features_per_layer[i+1],
                num_of_heads=num_heads_per_layer[i+1],
                concat=True if i < num_of_layers - 1 else False,  # last GAT layer does mean avg, the others do concat
                activation=nn.ELU() if i < num_of_layers - 1 else None,  # last layer just outputs raw scores
                dropout_prob=dropout,
                add_skip_connection=add_skip_connection,
                bias=bias,
                log_attention_weights=log_attention_weights
            )
            gat_layers.append(layer)

        self.gat_net = nn.Sequential(
            *gat_layers,
        )

    # data is just a (in_nodes_features, edge_index) tuple:

    def forward(self, data):
        return self.gat_net(data)

In [56]:

class DiffusionConvLayer(nn.Module):
    
    #At each diffusion step s, compute:
    #Z_+ = sum_{s=0}^{S-1} Î¸_s,1 Ã— (A_+_norm)^s Ã— X
    #Z_- = sum_{s=0}^{S-1} Î¸_s,2 Ã— (A_-_norm)^s Ã— X
    #Output: [Z_+ || Z_-] concatenated
    
    def __init__(self, in_features, out_channels, num_diffusion_steps=1, bias=True):
    #in_features: Input feature dimension = d
    #out_channels: Number of output channels per network (Q)
    #num_diffusion_steps: Number of diffusion steps (S)
        super().__init__()
        
        self.in_features = in_features
        self.out_channels = out_channels
        self.num_diffusion_steps = num_diffusion_steps
        
        # Learnable diffusion filters: one set for positive, one for negative
        # Shape: (S, d, Q) for each
        self.theta_pos = nn.Parameter(
            torch.Tensor(num_diffusion_steps, in_features, out_channels)
        )
        self.theta_neg = nn.Parameter(
            torch.Tensor(num_diffusion_steps, in_features, out_channels)
        )
        
        if bias:
            self.bias_pos = nn.Parameter(torch.Tensor(out_channels))
            self.bias_neg = nn.Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)
            
        self.reset_parameters()
        
    def reset_parameters(self):
        nn.init.xavier_uniform_(self.theta_pos)
        nn.init.xavier_uniform_(self.theta_neg)
        if self.bias_pos is not None:
            nn.init.zeros_(self.bias_pos)
        if self.bias_neg is not None:
            nn.init.zeros_(self.bias_neg)
    
    def forward(self, X_pos, X_neg, A_pos, A_neg):
        N = X_pos.shape[0] # X is N x d
        
        # Normalize adjacency matrices for random walk
        # D^(-1) Ã— A
        D_pos = A_pos.sum(dim=1, keepdim=True) + 1e-8  # (N, 1)
        D_neg = A_neg.sum(dim=1, keepdim=True) + 1e-8
        A_t_pos_norm = A_pos / D_pos  # (N, N)
        A_neg_norm = A_neg / D_neg
        
        # Initialize outputs
        Z_pos = torch.zeros(N, self.out_channels, device=X_pos.device) # 0's
        Z_neg = torch.zeros(N, self.out_channels, device=X_neg.device)

        # Positive diffusion
        A_power = torch.eye(N, device=X_pos.device)  # A^0 = Identity matrix
        for s in range(self.num_diffusion_steps):
            # Î¸_s Ã— (A^s Ã— X) for each output channel where s is the diffusion step
            # A_power: (N, N), X: (N, d), theta_pos[s]: (d, Q)
            diffused = A_power @ X_pos  # (N, d)
            Z_pos += diffused @ self.theta_pos[s]  # (N, d) @ (d, Q) = (N, Q) 
            A_power = A_power @ A_t_pos_norm  # Update A^s
        
        # Negative diffusion
        A_power = torch.eye(N, device=X_neg.device)
        for s in range(self.num_diffusion_steps):
            diffused = A_power @ X_neg
            Z_neg += diffused @ self.theta_neg[s]
            A_power = A_power @ A_neg_norm
        
        # Concatenate positive and negative outputs
        #Z = torch.cat([Z_pos, Z_neg], dim=1)  # (N, 2Q)

        if self.bias_pos is not None:
            Z_pos = Z_pos + self.bias_pos

        if self.bias_neg is not None:
            Z_neg = Z_neg + self.bias_neg

        return Z_pos, Z_neg # each is (N, Q)


In [57]:
class SpatialEncoder(nn.Module):
    
    def __init__(self, in_features_dim, hidden_channels, num_diffusion_steps=1, 
                 dropout=0.1):
        #hidden_channels: List of output channels for each layer, e.g., [32, 16] means 2 layers with 32 and 16 channels
        #num_diffusion_steps: Diffusion steps for each layer

        super().__init__()
        
        layers = []
        channels = [in_features_dim] + hidden_channels
        
        for i in range(len(channels) - 1):
            layers.append(
                DiffusionConvLayer(
                    in_features=channels[i],
                    out_channels=channels[i+1],
                    num_diffusion_steps=num_diffusion_steps
                )
            )
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout)) # dropout after each layer for regularization
        
        self.layers = nn.ModuleList(layers)
        
    def forward(self, Z_pos, Z_neg, A_pos, A_neg):

        #input Z_pos and Z_neg are just X at the start
        for layer in self.layers:
            if isinstance(layer, DiffusionConvLayer):
                Z_pos, Z_neg = layer(Z_pos, Z_neg, A_pos, A_neg)
            else:
                Z_pos = layer(Z_pos)
                Z_neg = layer(Z_neg)
        return Z_pos, Z_neg

class GraphConvGRUCell(nn.Module):



    def __init__(self, input_dim, hidden_dim, num_diffusion_steps=1):

        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Diffusion layers for reset gate
        self.conv_r = DiffusionConvLayer(
            in_features=input_dim + hidden_dim,
            out_channels=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        
        # Diffusion layers for update gate
        self.conv_u = DiffusionConvLayer(
            in_features=input_dim + hidden_dim,
            out_channels=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        
        # Diffusion layers for candidate activation
        self.conv_c = DiffusionConvLayer(
            in_features=input_dim + hidden_dim,
            out_channels=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        
    def forward(self, x_t_pos, x_t_neg, h_prev_pos, h_prev_neg, A_pos, A_neg):
        # Concatenate input and previous hidden state
        combined_pos = torch.cat([x_t_pos, h_prev_pos], dim=1)  # (N, input_dim + hidden_dim)
        combined_neg = torch.cat([x_t_neg, h_prev_neg], dim=1)  # (N, input_dim + hidden_dim)

        # Reset gate: r_t = sigmoid(DiffConv([x_t, h_{t-1}]))
        r_pos, r_neg = self.conv_r(combined_pos, combined_neg, A_pos, A_neg)  # (N, hidden_dim) each
        r_t_pos, r_t_neg = torch.sigmoid(r_pos), torch.sigmoid(r_neg)  # (N, hidden_dim) each

        # Update gate: u_t = sigmoid(DiffConv([x_t, h_{t-1}]))
        u_pos, u_neg = self.conv_u(combined_pos, combined_neg, A_pos, A_neg)
        u_t_pos, u_t_neg = torch.sigmoid(u_pos), torch.sigmoid(u_neg)  # (N, hidden_dim) each

        # Candidate: c_t = tanh(DiffConv([x_t, r_t * h_{t-1}]))
        #positive
        h_tilde_pos = r_t_pos * h_prev_pos  # Element-wise product
        h_tilde_neg = r_t_neg * h_prev_neg  # Element-wise product
        combined_c_pos = torch.cat([x_t_pos, h_tilde_pos], dim=1)
        combined_c_neg = torch.cat([x_t_neg, h_tilde_neg], dim=1)
        c_pos, c_neg = self.conv_c(combined_c_pos, combined_c_neg, A_pos, A_neg)
        c_t_pos, c_t_neg = torch.tanh(c_pos), torch.tanh(c_neg)  # (N, hidden_dim) each

        # Update hidden state: h_t = u_t * h_{t-1} + (1 - u_t) * c_t
        h_t_pos = u_t_pos * h_prev_pos + (1 - u_t_pos) * c_t_pos
        h_t_neg = u_t_neg * h_prev_neg + (1 - u_t_neg) * c_t_neg
        
        return h_t_pos, h_t_neg
    

In [58]:
class ReconstructionDecoder(nn.Module):
    
    def __init__(self, hidden_dim, feature_dim, use_structure_recon=True):

        super().__init__()
        
        self.use_structure_recon = use_structure_recon
        
        # Feature reconstruction: h_t â†’ X_t (pred)
        self.feature_decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, feature_dim)
        )
        
        # Structure reconstruction is done via inner product: A_hat = sigmoid(h Ã— h^T)
        # No parameters needed
        
    def forward(self, h_pos, h_neg, X_t, A_t_pos, A_t_neg):

        N = h_pos.shape[0]
        
        h_t = h_pos + h_neg
        # Reconstruct features
        X_hat = self.feature_decoder(h_t)  # (N, d)
        
        # Feature reconstruction error per node
        feature_error = torch.norm(X_t - X_hat, dim=1)  # (N,)
        
        # Reconstruct structure if requested
        if self.use_structure_recon and A_t_pos is not None and A_t_neg is not None:
            # Ã‚ = sigmoid(h Ã— h^T)
            A_hat_pos = torch.sigmoid(h_pos @ h_pos.T)  # (N, N)
            A_hat_neg = torch.sigmoid(h_neg @ h_neg.T)  # (N, N)
            
            # Structure reconstruction error per node (row-wise)
            struct_err_pos = torch.norm(A_t_pos - A_hat_pos, dim=1)  # (N,)
            struct_err_neg = torch.norm(A_t_neg - A_hat_neg, dim=1)  # (N,)
            structure_error = 0.5 * (struct_err_pos + struct_err_neg)

            loss_struct_pos = torch.mean((A_t_pos - A_hat_pos) ** 2)
            loss_struct_neg = torch.mean((A_t_neg - A_hat_neg) ** 2)
            loss_struct = 0.5 * (loss_struct_pos + loss_struct_neg)
            
            # Combined error (alpha balances feature vs structure)
            alpha = 1
            combined_error = alpha * feature_error + (1 - alpha) * structure_error
            
            # Compute losses
            loss_feat = torch.mean(feature_error ** 2)
            total_loss = alpha * loss_feat + (1 - alpha) * loss_struct
            
        else:
            A_hat_pos = None
            A_hat_neg = None
            combined_error = feature_error
            total_loss = torch.mean(feature_error ** 2)
        
        # Normalize errors to [0, 1] range (bubble signals per node)
        signals = (combined_error - combined_error.min()) / (combined_error.max() - combined_error.min() + 1e-8)
        #signals = combined_error
        return X_hat, A_hat_pos, A_hat_neg, total_loss, signals


class BubbleDetectionModel(nn.Module):
    def __init__(self, 
                 gat_config,
                 feature_dim,
                 embedding_dim,
                 encoder_channels, # list of channels for spatial encoder
                 hidden_dim,
                 num_diffusion_steps=1,
                 use_structure_recon=True,
                 dropout=0.1):

        super().__init__()
        
        self.feature_dim = feature_dim # d
        self.embedding_dim = embedding_dim # L - dim of PCA embeddings
        self.hidden_dim = hidden_dim # GRU hidden dim
        
        # 1. GAT for attention weights (you already have this)
        self.gat = GAT(
            num_of_layers=gat_config['num_layers'], # gat_config: Dict with GAT parameters (num_layers, heads, features)
            num_heads_per_layer=gat_config['heads'],
            num_features_per_layer=gat_config['features'],
            dropout=dropout
        )
        
        # 2. Spatial encoder
        self.spatial_encoder = SpatialEncoder(
            in_features_dim=feature_dim,
            hidden_channels=encoder_channels,
            num_diffusion_steps=num_diffusion_steps,
            dropout=dropout
        )
        
        # Output dimension of spatial encoder 
        spatial_out_dim = encoder_channels[-1]
        
        # 3. GRU cell (we'll add embedding_dim because we concatenate H_t)
        self.gru = GraphConvGRUCell(
            input_dim=spatial_out_dim + embedding_dim,
            hidden_dim=hidden_dim,
            num_diffusion_steps=num_diffusion_steps
        )
        
        # 4. Decoder
        self.decoder = ReconstructionDecoder(
            hidden_dim=hidden_dim,
            feature_dim=feature_dim,
            use_structure_recon=use_structure_recon
        )

    def forward(self, X_t, H_t, edge_index, corr_matrix, h_prev_pos=None, h_prev_neg=None, A_t_pos=None, A_t_neg=None):

        N = X_t.shape[0]
        
        # Initialise hidden state if None
        if h_prev_pos is None:
            h_prev_pos = torch.zeros(N, self.hidden_dim, device=X_t.device)
        if h_prev_neg is None:
            h_prev_neg = torch.zeros(N, self.hidden_dim, device=X_t.device)

        # Step 1: Compute attention weights with GAT
        # GAT expects (features, edge_index, corr_matrix)
        attention_weights = self.gat((H_t, edge_index, corr_matrix))  # (E, 2*NH, 1)
        
        # Convert edge attention to adjacency matrices
        A_pos, A_neg = self.attention_to_adjacency(
            attention_weights, edge_index, N
        )
        
        # Step 2: Spatial encoding
        Z_t_pos, Z_t_neg = self.spatial_encoder(X_t,X_t, A_pos, A_neg)  # (N, 2Q)
        
        # Step 3: Concatenate with PCA embeddings (bypass connection)
        Z_t_full_pos = torch.cat([Z_t_pos, H_t], dim=1)  # (N, 2Q + L)
        Z_t_full_neg = torch.cat([Z_t_neg, H_t], dim=1)  # (N, 2Q + L)
        
        # Step 4: GRU update
        h_t_pos, h_t_neg = self.gru(Z_t_full_pos, Z_t_full_neg, h_prev_pos, h_prev_neg, A_pos, A_neg)  # (N, hidden_dim)
        
        # Step 5: Decode and compute bubble signals
        X_hat, A_hat_pos, A_hat_neg, loss, bubble_signals = self.decoder(h_t_pos, h_t_neg, X_t, A_t_pos, A_t_neg)
        
        return h_t_pos, h_t_neg, bubble_signals, loss, A_pos, A_neg
    
    def attention_to_adjacency(self, attention_weights, edge_index, num_nodes):

        E = edge_index.shape[1] # (2, E -> 0 is src, 1 is trg)
        NH = attention_weights.shape[1] // 2 # (E, 2*NH, 1)
        
        # Average over attention heads
        att_pos = attention_weights[:, :NH, 0].mean(dim=1)  # (E,)
        att_neg = attention_weights[:, NH:, 0].mean(dim=1)  # (E,)
        
        # Create sparse adjacency matrices
        src = edge_index[0]
        trg = edge_index[1]
        
        A_pos = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        A_neg = torch.zeros(num_nodes, num_nodes, device=attention_weights.device)
        
        A_pos[src, trg] = att_pos
        A_neg[src, trg] = att_neg
        
        # Make symmetric -> graph is undirected
        A_pos = (A_pos + A_pos.T) / 2
        A_neg = (A_neg + A_neg.T) / 2
        
        return A_pos, A_neg


In [59]:
#since some stocks may have missing data, only consider stocks that have at least min_obs observations in the lookback window
# some stocks only added to SPX after certain date, so we only consider stocks that have enough data in the lookback window

def get_active_stocks(returns, t, lookback_days, feature_dfs=None, min_obs=21, eps=0.0):
    t = pd.to_datetime(t)
    window = returns.loc[t - pd.Timedelta(days=lookback_days): t]

    # enough non-NaN observations
    counts = window.notna().sum(axis=0)
    ok_obs = counts >= min_obs

    # not constant zero in the window (treat as missing asset)
    if eps == 0.0:
        ok_nonzero = ~(window.fillna(0.0) == 0.0).all(axis=0)
    else:
        ok_nonzero = ~(window.fillna(0.0).abs() <= eps).all(axis=0)

    #active = window.columns[ok_obs & ok_nonzero].tolist()
    active = window.columns[ok_nonzero].tolist()

    if feature_dfs is not None:
        active_set = set(active)
        for df in feature_dfs:
            # only care if the stock exists as a column in the dataframe
            if not df.empty:
                # Find intersection between current active stocks and this dataframe's columns
                active_set = active_set.intersection(df.columns)
        
        active = list(active_set)

    return active

def train_step(model, optimizer, X_t, H_t, edge_index, corr_matrix, A_t_pos, A_t_neg, h_prev_pos, h_prev_neg):
    #Single training step
    model.train()
    optimizer.zero_grad()

    h_t_pos, h_t_neg, signals, loss, A_pos, A_neg = model(X_t, H_t, edge_index, corr_matrix, h_prev_pos, h_prev_neg, A_t_pos, A_t_neg)

    loss.backward()
    optimizer.step()

    return loss.item(), h_t_pos.detach(), h_t_neg.detach(), signals.detach()


def inference_step(model, X_t, H_t, edge_index, corr_matrix, h_prev_pos, h_prev_neg):
    model.eval()
    with torch.no_grad():
        h_t_pos, h_t_neg, signals, _, A_t_pos, A_t_neg = model(
            X_t, H_t, edge_index, corr_matrix, h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg, A_t_pos=None, A_t_neg=None
        )
    return h_t_pos, h_t_neg, signals, A_t_pos, A_t_neg

In [60]:
def create_adjacency_from_correlation(corr_matrix, threshold=0.0):
    C = np.array(corr_matrix, dtype=np.float32)
    A_pos = np.maximum(C, 0.0)     
    A_neg = np.maximum(-C, 0.0)
    A_pos[A_pos < threshold] = 0 # min correlation to include edge
    A_neg[A_neg < threshold] = 0
    np.fill_diagonal(A_pos, 0) # self-loops
    np.fill_diagonal(A_neg, 0) # self-loops
    return torch.tensor(A_pos, dtype=torch.float32), torch.tensor(A_neg, dtype=torch.float32) # (N, N) is A

In [61]:
# functions for help 

from torch.optim import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import os
warnings.filterwarnings('ignore')
import traceback


def prepare_node_features(stocks, sectors, Z_DATA, t, norm_window=63): # 1 quarter
    """
    Normalise each feature per stock against its own rolling history (z-score),
    preserving signal relative to that stock's recent behaviour.
    """
    rows = []
    t = pd.to_datetime(t)
    
    for stock in stocks:
        sector_id = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
        
        # Simple lookup instead of rolling calculation
        feats = [sector_id]
        for key in Z_DATA:
            df = Z_DATA[key]
            val = df.loc[t, stock] if stock in df.columns and t in df.index else 0.0
            feats.append(float(val))
            
        rows.append(feats)

    features = np.array(rows, dtype=np.float32)
    return torch.tensor(np.nan_to_num(features), dtype=torch.float32)

# def prepare_node_features(stocks, sectors, volatility, market_caps, pe_ratios, implied_vol, short_interest,
#                            beta, operating_margin, return_on_equity, rsi_momentum, turnover, t):

#     rows = []
#     t = pd.to_datetime(t)
#     if not stocks:
#         return torch.empty((0, 2), dtype=torch.float32)
#     for stock in stocks:
#         # Get factors for stock at time t (or default values if missing)
#         sector_id = sectors.loc[stock, 'sector_id'] if stock in sectors.index else 0
#         market_cap = market_caps.loc[t, stock] if t in market_caps.index and stock in market_caps.columns else 0.0
#         pe_ratio = pe_ratios.loc[t, stock] if t in pe_ratios.index and stock in pe_ratios.columns else 0.0
#         implied_volatility = implied_vol.loc[t, stock] if t in implied_vol.index and stock in implied_vol.columns else 0.0
#         short_int = short_interest.loc[t, stock] if t in short_interest.index and stock in short_interest.columns else 0.0
#         beta_val = beta.loc[t, stock] if t in beta.index and stock in beta.columns else 0.0
#         op_margin = operating_margin.loc[t, stock] if t in operating_margin.index and stock in operating_margin.columns else 0.0
#         roe = return_on_equity.loc[t, stock] if t in return_on_equity.index and stock in return_on_equity.columns else 0.0
#         rsi = rsi_momentum.loc[t, stock] if t in rsi_momentum.index and stock in rsi_momentum.columns else 0.0
#         turn = turnover.loc[t, stock] if t in turnover.index and stock in turnover.columns else 0.0

#         # Get volatility at time t (or nearest available)
#         if t in volatility.index and stock in volatility.columns:
#             vol = volatility.loc[t, stock]
#         else:
#             # Get closest date
#             available_dates = volatility.index[volatility.index <= t]
#             if len(available_dates) > 0:
#                 closest_date = available_dates[-1]
#                 vol = volatility.loc[closest_date, stock]
#             else:
#                 vol = 0.0  # Default if no data available
#         rows.append([sector_id, vol, market_cap, pe_ratio, implied_volatility, short_int, beta_val, op_margin, roe, rsi, turn])
#         # features is (N, 11)
#     features = np.array(rows, dtype=np.float32)
#     features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)#


#     if len(features) >0:
#         num_cols = features.shape[1]
#         for i in range(num_cols):  # Normalize each feature to [0, 1]
#             if features[:, i].max() > features[:, i].min():
#                 features[:, i] = (features[:, i] - features[:, i].min()) / (features[:, i].max() - features[:, i].min() + 1e-8)

#     non_zero_count = np.count_nonzero(features)
#     total_elements = features.size
#     zero_fraction = 1.0 - (non_zero_count / total_elements)
    
#     if zero_fraction > 0.9: # If more than 90% of data is zero
#         print(f"\n[WARNING] Time {t}: {zero_fraction*100:.1f}% of features are ZERO.")
#         print("Sample Row (first stock):", features[0])
#         # Check raw dataframe lookup for one stock to debug
#         test_stock = stocks[0]
#         print(f"Debug check for {test_stock} at {t}:")
#         if test_stock in pe_ratios.columns:
#             # Check if exact date exists
#             date_exists = t in pe_ratios.index
#             print(f"  - Date {t} in PE_ratios index? {date_exists}")
#             if not date_exists:
#                 # Show nearest dates
#                 print(f"  - PE_ratios nearby dates: {pe_ratios.index[pe_ratios.index.get_indexer([t], method='nearest')]}")
            
#     return torch.tensor(features, dtype=torch.float32)


def create_edge_index(corr_matrix, k_neighbors=10):
    """
    Creates an edge index where each node connects to its top-k most correlated peers.
    """
    N = corr_matrix.shape[0]
    if isinstance(corr_matrix, np.ndarray):
        corr = torch.tensor(corr_matrix, dtype=torch.float32)
    else:
        corr = corr_matrix.clone()

    #diagonal to -infinity so a node doesn't select itself
    mask_diag = torch.eye(N, dtype=torch.bool, device=corr.device)
    corr.masked_fill_(mask_diag, float('-inf'))
    
    # includes strong positive and strong negative correlations
    vals, indices = torch.topk(corr.abs(), k=min(k_neighbors, N-1), dim=1)
    
    # create Edge List
    
    src_list = torch.arange(N, device=corr.device).repeat_interleave(k_neighbors)
    trg_list = indices.flatten()
    
    edge_index = torch.stack([src_list, trg_list], dim=0)
    
    return edge_index

In [62]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, save_path='checkpoint.pt'):
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.save_path)

In [63]:
def train_model_bo(
        model, optimizer, returns, sectors, volatility,
        Market_caps, PE_ratios, Implied_vol, Short_interest,
        Beta, Operating_margin, Return_on_equity, RSI_momentum,
        Turnover, Z_DATA, train_dates, stock2idx,
        patience=5, val_dates=None, embedding_cache=None,
        K=21, save_path='outputs/bo_ckpt.pt',
        k_neighbors=10,       # Round 3 param (fixed for Rounds 1-2)
        corr_threshold=0.0):  # Round 3 param (fixed for Rounds 1-2)
    """
    Refactored train_model accepting k_neighbors and corr_threshold explicitly
    so the same function works unchanged across all 3 BO rounds.
    Early stopping uses val reconstruction loss (fast proxy).
    Call compute_val_auc() separately to get the AUC objective.
    """
    device = next(model.parameters()).device
    N_full = len(stock2idx)
    early_stopping = EarlyStopping(patience=patience, save_path=save_path)
    feature_dfs_list = [
        volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover
    ]
    train_loss_history, val_loss_history = [], []

    for epoch in range(50):
        epoch_losses = []
        h_pos_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32).to(device)
        h_neg_full = torch.zeros(N_full, model.hidden_dim, dtype=torch.float32).to(device)

        for t in train_dates:
            active = get_active_stocks(returns, t, lookback_days=int(K * 1.5),
                                       feature_dfs=feature_dfs_list, min_obs=K)
            if not active:
                continue
            stocks_t  = active
            node_embs = embedding_cache.get(t, {})
            H_list = []
            for s in stocks_t:
                if s in node_embs:
                    e = node_embs[s]
                    H_list.append(e[0] if (len(e.shape) > 1 and e.shape[0] == 1) else e)
                else:
                    H_list.append(np.zeros(model.embedding_dim))
            H_t = torch.tensor(np.array(H_list), dtype=torch.float32).to(device)
            corr_matrix, _ = correlation_matrix(returns[stocks_t], t, K)
            corr_t     = torch.tensor(corr_matrix, dtype=torch.float32).to(device)
            edge_index = create_edge_index(corr_matrix, k_neighbors=k_neighbors)
            X_t        = prepare_node_features(stocks_t, sectors, Z_DATA, t).to(device)
            A_t_pos, A_t_neg = create_adjacency_from_correlation(corr_matrix,
                                                                  threshold=corr_threshold)
            A_t_pos, A_t_neg = A_t_pos.to(device), A_t_neg.to(device)
            active_idx = torch.tensor([stock2idx[s] for s in stocks_t],
                                      dtype=torch.long).to(device)
            h_prev_pos = h_pos_full.index_select(0, active_idx)
            h_prev_neg = h_neg_full.index_select(0, active_idx)

            model.train()
            optimizer.zero_grad()
            h_t_pos, h_t_neg, _, loss, _, _ = model(
                X_t, H_t, edge_index, corr_t,
                h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                A_t_pos=A_t_pos, A_t_neg=A_t_neg)
            if torch.isnan(loss) or torch.isinf(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            h_pos_full[active_idx] = h_t_pos.detach()
            h_neg_full[active_idx] = h_t_neg.detach()
            epoch_losses.append(loss.item())

        # ── Validation reconstruction loss (for early stopping) ───────────
        model.eval()
        val_losses = []
        with torch.no_grad():
            hv_pos = torch.zeros(N_full, model.hidden_dim).to(device)
            hv_neg = torch.zeros(N_full, model.hidden_dim).to(device)
            for t in val_dates:
                active = get_active_stocks(returns, t, lookback_days=int(K * 1.5),
                                           feature_dfs=feature_dfs_list, min_obs=K)
                if not active:
                    continue
                stocks_t  = active
                node_embs = embedding_cache.get(t, {})
                H_list = []
                for s in stocks_t:
                    if s in node_embs:
                        e = node_embs[s]
                        H_list.append(e[0] if (len(e.shape) > 1 and e.shape[0] == 1) else e)
                    else:
                        H_list.append(np.zeros(model.embedding_dim))
                H_t = torch.tensor(np.array(H_list), dtype=torch.float32).to(device)
                corr_matrix, _ = correlation_matrix(returns[stocks_t], t, K)
                corr_t     = torch.tensor(corr_matrix, dtype=torch.float32).to(device)
                edge_index = create_edge_index(corr_matrix, k_neighbors=k_neighbors)
                X_t        = prepare_node_features(stocks_t, sectors, Z_DATA, t).to(device)
                active_idx = torch.tensor([stock2idx[s] for s in stocks_t],
                                          dtype=torch.long).to(device)
                hv_prev_pos = hv_pos.index_select(0, active_idx)
                hv_prev_neg = hv_neg.index_select(0, active_idx)
                ht_pos, ht_neg, _, loss, _, _ = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=hv_prev_pos, h_prev_neg=hv_prev_neg)
                hv_pos[active_idx] = ht_pos.detach()
                hv_neg[active_idx] = ht_neg.detach()
                val_losses.append(loss.item())

        avg_train = np.mean(epoch_losses) if epoch_losses else float('inf')
        avg_val   = np.mean(val_losses)   if val_losses   else float('inf')
        train_loss_history.append(avg_train)
        val_loss_history.append(avg_val)
        print(f'  Epoch {epoch+1}: train={avg_train:.4f} val={avg_val:.4f}')

        early_stopping(avg_val, model)
        if early_stopping.early_stop:
            print('  Early stopping triggered.')
            break

    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=False))
    return model, train_loss_history, val_loss_history


In [64]:
def test_model(model, returns, sectors, volatility, Market_caps, PE_ratios,
               Implied_vol, Short_interest, Beta, Operating_margin,
               Return_on_equity, RSI_momentum, Turnover, Z_DATA,
               embedding_cache, test_dates, stock2idx,
               K=21, k_neighbors=10, corr_threshold=0.0):
    """Returns dict {date: (active_stocks_list, signals_np_array)}"""
    device = next(model.parameters()).device
    N_full = len(stock2idx)
    h_pos_full = torch.zeros(N_full, model.hidden_dim, device=device)
    h_neg_full = torch.zeros(N_full, model.hidden_dim, device=device)
    feature_dfs_list = [
        volatility, Market_caps, PE_ratios, Implied_vol, Short_interest,
        Beta, Operating_margin, Return_on_equity, RSI_momentum, Turnover
    ]
    all_signals, valid_dates, active_lists = [], [], []
    for t in test_dates:
        try:
            active = get_active_stocks(returns, t, lookback_days=int(K * 1.5),
                                       feature_dfs=feature_dfs_list, min_obs=K)
            if not active:
                continue
            stocks_t  = active
            node_embs = embedding_cache.get(t, {})
            H_list = []
            for s in stocks_t:
                if s in node_embs:
                    H_list.append(node_embs[s])
                else:
                    H_list.append(np.zeros(model.embedding_dim))
            H_t = torch.tensor(np.array(H_list), dtype=torch.float32).to(device)
            corr_matrix, _ = correlation_matrix(returns[stocks_t], t, K)
            corr_t     = torch.tensor(corr_matrix, dtype=torch.float32).to(device)
            edge_index = create_edge_index(corr_matrix, k_neighbors=k_neighbors)
            X_t        = prepare_node_features(stocks_t, sectors, Z_DATA, t).to(device)
            active_idx = torch.tensor([stock2idx[s] for s in stocks_t],
                                      dtype=torch.long).to(device)
            h_prev_pos = h_pos_full.index_select(0, active_idx)
            h_prev_neg = h_neg_full.index_select(0, active_idx)
            model.eval()
            with torch.no_grad():
                h_t_pos, h_t_neg, signals, _, _, _ = model(
                    X_t, H_t, edge_index, corr_t,
                    h_prev_pos=h_prev_pos, h_prev_neg=h_prev_neg,
                    A_t_pos=None, A_t_neg=None)
            h_pos_full[active_idx] = h_t_pos.detach()
            h_neg_full[active_idx] = h_t_neg.detach()
            all_signals.append(signals.cpu().numpy())
            valid_dates.append(t)
            active_lists.append(stocks_t)
        except Exception:
            traceback.print_exc()
            continue
    return {d: (active_lists[i], all_signals[i]) for i, d in enumerate(valid_dates)}


In [65]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import torch


def evaluate_once(
        test_results,
        prices,
        forward_window,
        crash_threshold
):

    y_true = []
    y_scores = []

    sorted_dates = sorted(test_results.keys())

    valid_dates = [
        d for d in sorted_dates
        if d <= prices.index[-1] - pd.Timedelta(days=forward_window)
    ]

    for t in valid_dates:

        stocks, signals = test_results[t]

        if isinstance(signals, torch.Tensor):
            signals = signals.cpu().numpy()

        signals = signals.flatten()

        available = [s for s in stocks if s in prices.columns]

        if len(available) == 0:
            continue

        mask = [i for i,s in enumerate(stocks) if s in available]

        signals = signals[mask]

        p_t = prices.loc[t, available]

        future_idx = prices.index.searchsorted(
            t + pd.Timedelta(days=forward_window)
        )

        if future_idx >= len(prices):
            continue

        future_date = prices.index[future_idx]

        p_future = prices.loc[future_date, available]

        fwd_returns = (p_future - p_t) / p_t

        crash = (fwd_returns < crash_threshold).astype(int)

        y_true.extend(crash.values)

        y_scores.extend(signals)

    if len(y_true) == 0:
        return None

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    auc = roc_auc_score(y_true, y_scores)

    baseline = y_true.mean()

    precision = y_true[y_scores > np.percentile(y_scores, 90)].mean()

    lift = precision / baseline if baseline > 0 else np.nan

    return auc, lift, baseline

def grid_search(
        test_results,
        prices,
        forward_windows,
        crash_thresholds
):

    rows = []

    for fw in forward_windows:

        for ct in crash_thresholds:

            result = evaluate_once(
                test_results,
                prices,
                fw,
                ct
            )

            if result is None:
                continue

            auc, lift, baseline = result

            rows.append({

                "ForwardWindow": fw,

                "CrashThreshold": ct,

                "AUC": auc,

                "Lift": lift,

                "Baseline": baseline

            })

            print(
                f"FW={fw:3d} "
                f"CT={ct:6.2f} "
                f"AUC={auc:.3f} "
                f"Lift={lift:.2f}"
            )

    return pd.DataFrame(rows)


In [66]:
# ================================================================
# DATA LOADING  —  run ONCE before any BO round
# ================================================================
print('[1/4] Loading prices and returns...')
prices = pd.read_excel('SPX_sectors_data.xlsx', header=[0, 1], index_col=0)
prices.dropna(how='all', inplace=True)
prices = prices.ffill().bfill()
prices.columns = prices.columns.droplevel(1)

all_stocks = prices.columns.tolist()
stock2idx  = {s: i for i, s in enumerate(all_stocks)}
N_full     = len(all_stocks)

sectors = pd.read_excel('SPX_sectors_data.xlsx', sheet_name='Sectors', header=0, index_col=0)
sectors['sector_id'] = sectors['Sector'].astype('category').cat.codes

returns = pd.read_excel('SPX_sectors_data.xlsx', header=[0, 1], index_col=0)
returns.columns = returns.columns.get_level_values(0)
returns.dropna(how='all', inplace=True)
returns = returns.pct_change().dropna(how='all')

# Data splits — test set is held out during all BO rounds
train_returns = returns.loc['2012-01-01':'2016-12-31'].ffill().bfill()
val_returns   = returns.loc['2017-01-01':'2019-06-30'].ffill().bfill()
test_returns  = returns.loc['2019-07-01':'2024-12-31'].ffill().bfill()
train_prices  = prices.loc['2012-01-01':'2016-12-31']
val_prices    = prices.loc['2017-01-01':'2019-06-30']
test_prices   = prices.loc['2019-07-01':'2024-12-31']

# Full returns used for rolling vol context across the whole history
full_returns  = returns.ffill().bfill()
all_vol       = full_returns.rolling(window=21).std() * np.sqrt(252)

def load_and_fix_index(filename):
    df = pd.read_csv(filename, index_col=0)
    df.index = pd.to_datetime(df.index, format='%m/%d/%Y')
    df.dropna(how='all', inplace=True)
    return df.ffill().bfill()

print('[2/4] Loading constituent factors...')
Market_caps      = load_and_fix_index('Data/SPX_Constituents_market_cap_2006_2025(in).csv')
PE_ratios        = load_and_fix_index('Data/SPX_Constituents_Calculated_PE_2006_2025(in).csv')
Implied_vol      = load_and_fix_index('Data/SPX_Constituents_Implied_vol_2006_2025(in).csv')
Beta             = load_and_fix_index('Data/SPX_Constituents_Beta_2006_2025(in).csv')
Operating_margin = load_and_fix_index('Data/SPX_Constituents_Op_Margin_2006_2025(in).csv')
Return_on_equity = load_and_fix_index('Data/SPX_Constituents_Ret_On_Equity_2006_2025(in).csv')
RSI_momentum     = load_and_fix_index('Data/SPX_Constituents_RSI_momentum_2006_2025(in).csv')
Short_interest   = load_and_fix_index('Data/SPX_Constituents_Short_Interest_Pct_2006_2025(in).csv')
Turnover         = load_and_fix_index('Data/SPX_Constituents_Turnover_30D_2006_2025(in).csv')

print('[3/4] Precomputing Z-scores...')
def precompute_zscores(df, window=63):
    mu  = df.rolling(window=window, min_periods=5).mean()
    sig = df.rolling(window=window, min_periods=5).std()
    return ((df - mu) / (sig + 1e-8)).clip(-5.0, 5.0).fillna(0.0)

Z_DATA = {
    'volatility':     precompute_zscores(all_vol),
    'market_caps':    precompute_zscores(Market_caps),
    'pe_ratios':      precompute_zscores(PE_ratios),
    'implied_vol':    precompute_zscores(Implied_vol),
    'short_interest': precompute_zscores(Short_interest),
    'beta':           precompute_zscores(Beta),
    'op_margin':      precompute_zscores(Operating_margin),
    'roe':            precompute_zscores(Return_on_equity),
    'rsi':            precompute_zscores(RSI_momentum),
    'turnover':       precompute_zscores(Turnover),
}
FEATURE_DIM = 11  # sector_id + 10 factor z-scores
all_dates = sorted(
    set(train_returns.index) | set(val_returns.index) | set(test_returns.index)
)
print(f'[4/4] Done.')
print(f'  Train: {train_returns.index[0].date()} to {train_returns.index[-1].date()}')
print(f'  Val:   {val_returns.index[0].date()} to {val_returns.index[-1].date()}')
print(f'  Test:  {test_returns.index[0].date()} to {test_returns.index[-1].date()} (held out)')


[1/4] Loading prices and returns...
[2/4] Loading constituent factors...
[3/4] Precomputing Z-scores...
[4/4] Done.
  Train: 2012-01-03 to 2016-12-30
  Val:   2017-01-03 to 2019-06-28
  Test:  2019-07-01 to 2024-12-31 (held out)


In [67]:
# ================================================================
# BO HELPERS  —  run once after data loading
# ================================================================
SEED = 42  # single seed during BO for speed; final training uses 4 seeds

# Validation AUC is averaged across these two (forward_window, crash_threshold) combos
VAL_FW1, VAL_CT1 = 22, -0.20
VAL_FW2, VAL_CT2 = 22, -0.30


def make_embedding_cache(K):
    """Precompute PCA embeddings for all dates with lookback window K."""
    combined = pd.concat([train_returns, val_returns, test_returns]).sort_index()
    #combined = combined[~combined.index.duplicated()].ffill().bfill()
    combined = combined[~combined.index.duplicated(keep='first')]
    cache = {}
    for t in tqdm(all_dates, desc=f'Embeddings K={K}', leave=False):
        embs, _ = compute_initial_node_embeddings(combined, t, K)
        cache[t] = embs
    return cache


def compute_val_auc(val_results, val_prices_df,
                    fw1=VAL_FW1, ct1=VAL_CT1, fw2=VAL_FW2, ct2=VAL_CT2):
    """Average AUC across two (forward_window, crash_threshold) settings."""
    aucs = []
    for fw, ct in [(fw1, ct1), (fw2, ct2)]:
        result = evaluate_once(val_results, val_prices_df, fw, ct)
        if result is not None:
            auc, _, _ = result
            aucs.append(auc)
    return float(np.mean(aucs)) if aucs else 0.5


def make_date_ranges(K):
    """Return (train_dates, val_dates) adjusted for lookback window K."""
    td = train_returns.loc[
        train_returns.index[0] + pd.Timedelta(days=K * 2):].index
    vd = val_returns.loc[
        val_returns.index[0] + pd.Timedelta(days=K * 2):].index
    return td, vd


print('BO helpers ready.')
print(f'  Val AUC = mean of AUC(FW={VAL_FW1}, CT={VAL_CT1}) '
      f'and AUC(FW={VAL_FW2}, CT={VAL_CT2})')


BO helpers ready.
  Val AUC = mean of AUC(FW=22, CT=-0.2) and AUC(FW=22, CT=-0.3)


In [ ]:
# ================================================================
# ROUND 1 — Training dynamics (ACTIVE)
# Tuning:  lr, weight_decay, K (10-44), patience
# Fixed:   all architecture and graph params at original defaults
# ================================================================

# Fixed architecture for Round 1
R1 = dict(
    embedding_dim=10, gat_output_dim=8, gat_heads=1,
    encoder_channels=[16, 8], hidden_dim=32,
    diffusion_steps=1, dropout=0.1,
    k_neighbors=10, corr_threshold=0.0,
)
_r1_emb_cache = {}  # reuse embeddings across trials with the same K


def r1_objective(trial):
    lr           = trial.suggest_float('lr',           1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    K            = trial.suggest_int(  'K',            10,   44)
    patience     = trial.suggest_int(  'patience',     3,    15)

    print(f'  Trial {trial.number}: lr={lr:.2e}  wd={weight_decay:.2e}  '
          f'K={K}  patience={patience}')

    # Recompute embeddings only when K changes
    if K not in _r1_emb_cache:
        print(f'    Precomputing embeddings for K={K}...')
        _r1_emb_cache[K] = make_embedding_cache(K)
    emb_cache = _r1_emb_cache[K]

    train_dates, val_dates = make_date_ranges(K)

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    gat_config = {
        'num_layers': 1,
        'heads':      [R1['gat_heads']],
        'features':   [R1['embedding_dim'], R1['gat_output_dim']],
    }
    model = BubbleDetectionModel(
        gat_config=gat_config,
        feature_dim=FEATURE_DIM,
        embedding_dim=R1['embedding_dim'],
        encoder_channels=R1['encoder_channels'],
        hidden_dim=R1['hidden_dim'],
        num_diffusion_steps=R1['diffusion_steps'],
        use_structure_recon=True,
        dropout=R1['dropout'],
    )
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    save_path = f'outputs/bo_r1_t{trial.number}.pt'
    os.makedirs('outputs', exist_ok=True)

    try:
        model, _, _ = train_model_bo(
            model, optimizer, full_returns, sectors, all_vol,
            Market_caps, PE_ratios, Implied_vol, Short_interest,
            Beta, Operating_margin, Return_on_equity, RSI_momentum,
            Turnover, Z_DATA, train_dates, stock2idx,
            patience=patience, val_dates=val_dates,
            embedding_cache=emb_cache, K=K, save_path=save_path,
            k_neighbors=R1['k_neighbors'],
            corr_threshold=R1['corr_threshold'],
        )
    except Exception as e:
        print(f'    Training failed: {e}')
        return 0.5

    try:
        val_results = test_model(
            model, full_returns, sectors, all_vol,
            Market_caps, PE_ratios, Implied_vol, Short_interest,
            Beta, Operating_margin, Return_on_equity, RSI_momentum,
            Turnover, Z_DATA, emb_cache, val_dates, stock2idx,
            K=K,
            k_neighbors=R1['k_neighbors'],
            corr_threshold=R1['corr_threshold'],
        )
        val_auc = compute_val_auc(val_results, val_prices)
    except Exception as e:
        print(f'    Evaluation failed: {e}')
        val_auc = 0.5

    if os.path.exists(save_path):
        os.remove(save_path)  # clean up checkpoint to save disk

    print(f'    => Val AUC: {val_auc:.4f}')
    return val_auc


N_TRIALS_R1 = 40  # 40-60 recommended; first 10 are random warm-up, rest are TPE
r1_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=0, n_startup_trials=10),
    study_name='round1_training_dynamics',
)
print(f'Starting Round 1 Bayesian Optimisation ({N_TRIALS_R1} trials)')
print(f'Objective: avg AUC(FW={VAL_FW1}, CT={VAL_CT1}) and AUC(FW={VAL_FW2}, CT={VAL_CT2})')
print('-' * 60)
r1_study.optimize(r1_objective, n_trials=N_TRIALS_R1)

r1_best     = r1_study.best_params
r1_best_auc = r1_study.best_value
print('\n' + '=' * 60)
print(f'ROUND 1 COMPLETE  |  Best Val AUC = {r1_best_auc:.4f}')
for k, v in r1_best.items():
    print(f'  {k} = {v}')


Starting Round 1 Bayesian Optimisation (40 trials)
Objective: avg AUC(FW=22, CT=-0.2) and AUC(FW=22, CT=-0.3)
------------------------------------------------------------
  Trial 0: lr=1.25e-03  wd=1.40e-04  K=31  patience=10
    Precomputing embeddings for K=31...


  Epoch 1: train=10.3090 val=9.2588
  Epoch 2: train=4.7598 val=15.6132
  Epoch 3: train=3.5857 val=4.0691
  Epoch 4: train=2.8811 val=4.6256
  Epoch 5: train=2.4571 val=3.5390
  Epoch 6: train=2.1901 val=5.7211
  Epoch 7: train=1.9738 val=4.8898
  Epoch 8: train=1.7695 val=4.4655
  Epoch 9: train=1.6095 val=4.4519
  Epoch 10: train=1.5091 val=4.3121
  Epoch 11: train=1.4557 val=4.1380
  Epoch 12: train=1.4003 val=3.8278
  Epoch 13: train=1.3807 val=3.7003
  Epoch 14: train=1.3863 val=3.3111
  Epoch 15: train=1.3889 val=3.0981
  Epoch 16: train=1.4373 val=3.2344
  Epoch 17: train=1.4309 val=4.0354
  Epoch 18: train=1.5045 val=4.0894
  Epoch 19: train=1.3575 val=4.7511
  Epoch 20: train=1.2373 val=3.6888
  Epoch 21: train=1.1489 val=4.1225
  Epoch 22: train=1.1622 val=3.8857
  Epoch 23: train=1.0995 val=6.2859
  Epoch 24: train=1.0811 val=4.6947
  Epoch 25: train=1.0315 val=9.9446
  Early stopping triggered.
    => Val AUC: 0.4785
  Trial 1: lr=7.04e-04  wd=8.66e-05  K=25  patience=14
 

  Epoch 1: train=12.1949 val=8.3632
  Epoch 2: train=5.5792 val=6.0808
  Epoch 3: train=3.9282 val=3.9132
  Epoch 4: train=3.2396 val=3.3349
  Epoch 5: train=2.8177 val=3.0993
  Epoch 6: train=2.5965 val=3.4709
  Epoch 7: train=2.3777 val=2.6816
  Epoch 8: train=2.2315 val=2.9577
  Epoch 9: train=2.0882 val=3.7593
  Epoch 10: train=1.9887 val=3.2627
  Epoch 11: train=1.8350 val=3.7156
  Epoch 12: train=1.7410 val=3.9335
  Epoch 13: train=1.6797 val=3.3035
  Epoch 14: train=1.5687 val=5.5400
  Epoch 15: train=1.5285 val=4.5623
  Epoch 16: train=1.4158 val=4.5681
  Epoch 17: train=1.3560 val=3.5646
  Epoch 18: train=1.3004 val=3.0531
  Epoch 19: train=1.2739 val=2.6341
  Epoch 20: train=1.2418 val=3.4750
  Epoch 21: train=1.2106 val=3.3727
  Epoch 22: train=1.1686 val=2.4248
  Epoch 23: train=1.1519 val=2.9829
  Epoch 24: train=1.1237 val=4.3442
  Epoch 25: train=1.0935 val=2.3455
  Epoch 26: train=1.0820 val=2.9253
  Epoch 27: train=1.0711 val=3.0930
  Epoch 28: train=1.0171 val=3.2424


  Epoch 1: train=6.7273 val=34.1979
  Epoch 2: train=3.6655 val=5.2218
  Epoch 3: train=2.2748 val=6.6052
  Epoch 4: train=1.8902 val=8.4484
  Epoch 5: train=1.8268 val=5.3128
  Epoch 6: train=1.6349 val=4.0952
  Epoch 7: train=1.4360 val=3.3780
  Epoch 8: train=1.2650 val=8.9587
  Epoch 9: train=1.1373 val=4.4186
  Epoch 10: train=1.1311 val=3.8723
  Epoch 11: train=1.1757 val=9.7805
  Epoch 12: train=1.2234 val=5.4301
  Epoch 13: train=1.3176 val=4.4160
  Epoch 14: train=1.1303 val=10.3325
  Epoch 15: train=1.1219 val=8.7037
  Epoch 16: train=1.1617 val=8.0322
  Early stopping triggered.
    => Val AUC: 0.5526
  Trial 3: lr=1.37e-03  wd=5.98e-04  K=12  patience=4
    Precomputing embeddings for K=12...


  Epoch 1: train=11.2142 val=8.6630
  Epoch 2: train=6.3801 val=5.1538
  Epoch 3: train=3.9685 val=4.1049
  Epoch 4: train=3.1594 val=6.7301
  Epoch 5: train=2.7647 val=9.3998
  Epoch 6: train=2.6359 val=6.3913
  Epoch 7: train=2.4654 val=3.8632
  Epoch 8: train=2.3688 val=4.9239
  Epoch 9: train=2.2334 val=10.1244
  Epoch 10: train=2.0939 val=3.8639
  Epoch 11: train=1.9392 val=4.2772
  Early stopping triggered.
    => Val AUC: 0.6046
  Trial 4: lr=1.10e-04  wd=3.15e-04  K=37  patience=14
  Epoch 1: train=24.1666 val=16.6381
  Epoch 2: train=13.2617 val=10.9339
  Epoch 3: train=9.7533 val=8.7878
  Epoch 4: train=8.1775 val=7.7140
  Epoch 5: train=7.6350 val=7.3480
  Epoch 6: train=7.1191 val=6.3981
  Epoch 7: train=6.2294 val=5.8175
  Epoch 8: train=5.8972 val=5.6478
  Epoch 9: train=5.6464 val=5.5324
  Epoch 10: train=5.2629 val=4.7723
  Epoch 11: train=4.7757 val=4.3697
  Epoch 12: train=4.4552 val=3.9832
  Epoch 13: train=4.2680 val=3.7370
  Epoch 14: train=4.0767 val=3.5973
  Epoc

...

  Epoch 1: train=18.2882 val=10.6807
  Epoch 2: train=8.7254 val=7.8098
  Epoch 3: train=7.5296 val=7.4701
  Epoch 4: train=7.0592 val=6.3133
  Epoch 5: train=5.8749 val=5.4776
  Epoch 6: train=5.3430 val=4.9074
  Epoch 7: train=4.5932 val=3.7575
  Epoch 8: train=3.9223 val=3.4672
  Epoch 9: train=3.6932 val=3.2242
  Epoch 10: train=3.5082 val=3.0199
  Epoch 11: train=3.3287 val=2.9156
  Epoch 12: train=3.1509 val=2.6029
  Epoch 13: train=2.9969 val=3.5796
  Epoch 14: train=2.8497 val=2.3462
  Epoch 15: train=2.7747 val=3.0654
  Epoch 16: train=2.7273 val=2.7475
  Epoch 17: train=2.6248 val=3.3968
  Epoch 18: train=2.5600 val=2.6963
  Epoch 19: train=2.5344 val=3.7706
  Early stopping triggered.
    => Val AUC: 0.6398
  Trial 24: lr=1.61e-04  wd=2.16e-04  K=35  patience=4
    Precomputing embeddings for K=35...


  Epoch 1: train=21.0989 val=13.2632
  Epoch 2: train=10.3554 val=8.6925
  Epoch 3: train=8.0262 val=7.6567
  Epoch 4: train=7.4582 val=7.1687
  Epoch 5: train=6.4829 val=5.9285
  Epoch 6: train=5.7836 val=5.5248
  Epoch 7: train=5.4755 val=5.7829
  Epoch 8: train=5.2267 val=5.2181
  Epoch 9: train=4.7770 val=4.1579
  Epoch 10: train=4.1283 val=3.7739
  Epoch 11: train=3.7520 val=3.2795
  Epoch 12: train=3.4171 val=3.2969
  Epoch 13: train=3.2480 val=2.6307
  Epoch 14: train=3.1005 val=2.8285
  Epoch 15: train=3.0166 val=2.3469
  Epoch 16: train=2.8850 val=2.8209
  Epoch 17: train=2.8188 val=2.2280
  Epoch 18: train=2.7020 val=2.5484
  Epoch 19: train=2.6284 val=2.5484
  Epoch 20: train=2.5332 val=1.9123
  Epoch 21: train=2.4724 val=1.7936
  Epoch 22: train=2.3523 val=1.7650
  Epoch 23: train=2.3392 val=2.0797
  Epoch 24: train=2.3050 val=1.6854
  Epoch 25: train=2.2325 val=1.9053
  Epoch 26: train=2.2031 val=2.5908
  Epoch 27: train=2.1726 val=2.6571
  Epoch 28: train=2.1452 val=2.169

  Epoch 1: train=14.8915 val=8.3362
  Epoch 2: train=7.5113 val=7.6387
  Epoch 3: train=6.4148 val=5.7288
  Epoch 4: train=4.6747 val=5.3100
  Epoch 5: train=4.1524 val=3.7009
  Epoch 6: train=3.7660 val=3.4403
  Epoch 7: train=3.4960 val=3.1832
  Epoch 8: train=3.1974 val=2.6704
  Epoch 9: train=2.8935 val=3.6528
  Epoch 10: train=2.6833 val=2.6210
  Epoch 11: train=2.5547 val=2.3247
  Epoch 12: train=2.4963 val=2.5423
  Epoch 13: train=2.4348 val=3.2811
  Epoch 14: train=2.3447 val=3.1662
  Early stopping triggered.
    => Val AUC: 0.6163
  Trial 27: lr=1.53e-04  wd=1.50e-04  K=29  patience=8
  Epoch 1: train=21.3911 val=13.6402
  Epoch 2: train=10.5718 val=8.8470
  Epoch 3: train=8.0926 val=7.6078
  Epoch 4: train=7.1234 val=6.4038
  Epoch 5: train=6.0840 val=5.8186
  Epoch 6: train=5.6631 val=6.0309
  Epoch 7: train=5.3823 val=5.3937
  Epoch 8: train=4.9426 val=4.4618
  Epoch 9: train=4.5233 val=4.0334
  Epoch 10: train=4.0751 val=3.5372
  Epoch 11: train=3.5899 val=2.8946
  Epoch 

  Epoch 1: train=17.7843 val=9.9092
  Epoch 2: train=8.3105 val=7.6485
  Epoch 3: train=7.4585 val=7.4978
  Epoch 4: train=7.0767 val=6.7659
  Epoch 5: train=6.5110 val=6.1291
  Epoch 6: train=5.8884 val=5.4438
  Epoch 7: train=5.1269 val=4.8096
  Epoch 8: train=4.6001 val=4.2635
  Epoch 9: train=4.3256 val=4.0965
  Epoch 10: train=3.9360 val=3.6132
  Epoch 11: train=3.5569 val=3.0344
  Epoch 12: train=3.2972 val=2.8893
  Epoch 13: train=3.0966 val=2.4651
  Epoch 14: train=2.9477 val=2.3405
  Epoch 15: train=2.7524 val=2.7213
  Epoch 16: train=2.6316 val=2.3383
  Epoch 17: train=2.5640 val=2.1297
  Epoch 18: train=2.4453 val=1.9523
  Epoch 19: train=2.4217 val=1.9909
  Epoch 20: train=2.3588 val=1.8278
  Epoch 21: train=2.2797 val=1.8980
  Epoch 22: train=2.2620 val=1.8085
  Epoch 23: train=2.2051 val=2.1411
  Epoch 24: train=2.1641 val=2.1447
  Epoch 25: train=2.1297 val=2.1118
  Epoch 26: train=2.1048 val=3.3155
  Epoch 27: train=2.0631 val=2.6284
  Epoch 28: train=2.0004 val=8.6638


  Epoch 1: train=9.2014 val=8.9723


In [ ]:
# Round 1 — Results analysis
r1_df = r1_study.trials_dataframe()[
    ['number', 'value', 'params_lr', 'params_weight_decay', 'params_K', 'params_patience']
].copy()
r1_df.columns = ['trial', 'val_auc', 'lr', 'weight_decay', 'K', 'patience']
r1_df = r1_df.sort_values('val_auc', ascending=False).reset_index(drop=True)
print(r1_df.head(10).to_string(index=False))

try:
    importances = optuna.importance.get_param_importances(r1_study)
    print('\nParam importances:', {k: f'{v:.3f}' for k, v in importances.items()})
except Exception:
    pass

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ['lr', 'weight_decay', 'K', 'patience']):
    ax.scatter(r1_df[col], r1_df['val_auc'], alpha=0.6, s=30)
    ax.axvline(r1_best[col], color='red', linestyle='--', linewidth=1.5,
               label=f"best={r1_best[col]:.3g}")
    ax.set_xlabel(col)
    ax.set_ylabel('Val AUC')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.suptitle('Round 1: Param vs Val AUC', y=1.02)
plt.tight_layout()
plt.show()

print('\n>>> Copy these into Round 2 / final training:')
print(f"  lr           = {r1_best['lr']:.6f}")
print(f"  weight_decay = {r1_best['weight_decay']:.2e}")
print(f"  K            = {r1_best['K']}")
print(f"  patience     = {r1_best['patience']}")


In [ ]:
# ================================================================
# ROUND 2 — Architecture capacity  (UNCOMMENT TO RUN)
# Paste Round 1 best params into R2_* variables, then uncomment.
# ================================================================

# R2_LR       = r1_best['lr']           # or paste: e.g. 5e-4
# R2_WD       = r1_best['weight_decay']
# R2_K        = r1_best['K']
# R2_PATIENCE = r1_best['patience']
# _r2_emb_cache = {}

# def r2_objective(trial):
#     embedding_dim   = trial.suggest_int(  'embedding_dim',    6,  20)
#     gat_output_dim  = trial.suggest_int(  'gat_output_dim',   4,  16)
#     gat_heads       = trial.suggest_categorical('gat_heads',  [1, 2, 4])
#     hidden_dim      = trial.suggest_categorical('hidden_dim', [16, 32, 64, 128])
#     enc_ch1         = trial.suggest_categorical('enc_ch1',    [8, 16, 32, 64])
#     enc_ch2         = trial.suggest_categorical('enc_ch2',    [4,  8, 16, 32])
#     diffusion_steps = trial.suggest_int(  'diffusion_steps',  1,  4)
#     dropout         = trial.suggest_float('dropout',          0.0, 0.5)
#     if enc_ch2 > enc_ch1:  # shrinking encoder constraint
#         raise optuna.exceptions.TrialPruned()
#
#     print(f'  Trial {trial.number}: emb={embedding_dim} gat_out={gat_output_dim} '
#           f'heads={gat_heads} hid={hidden_dim} enc=[{enc_ch1},{enc_ch2}] '
#           f'diff={diffusion_steps} drop={dropout:.2f}')
#
#     K = R2_K
#     if K not in _r2_emb_cache:
#         _r2_emb_cache[K] = make_embedding_cache(K)
#     emb_cache = _r2_emb_cache[K]
#     train_dates, val_dates = make_date_ranges(K)
#
#     torch.manual_seed(SEED); np.random.seed(SEED)
#     gat_config = {'num_layers': 1, 'heads': [gat_heads],
#                   'features': [embedding_dim, gat_output_dim]}
#     model = BubbleDetectionModel(
#         gat_config=gat_config, feature_dim=FEATURE_DIM,
#         embedding_dim=embedding_dim, encoder_channels=[enc_ch1, enc_ch2],
#         hidden_dim=hidden_dim, num_diffusion_steps=diffusion_steps,
#         use_structure_recon=True, dropout=dropout,
#     )
#     optimizer = Adam(model.parameters(), lr=R2_LR, weight_decay=R2_WD)
#     save_path = f'outputs/bo_r2_t{trial.number}.pt'
#     os.makedirs('outputs', exist_ok=True)
#     try:
#         model, _, _ = train_model_bo(
#             model, optimizer, full_returns, sectors, all_vol,
#             Market_caps, PE_ratios, Implied_vol, Short_interest,
#             Beta, Operating_margin, Return_on_equity, RSI_momentum,
#             Turnover, Z_DATA, train_dates, stock2idx,
#             patience=R2_PATIENCE, val_dates=val_dates,
#             embedding_cache=emb_cache, K=K, save_path=save_path,
#             k_neighbors=10, corr_threshold=0.0,
#         )
#     except Exception as e:
#         print(f'    Training failed: {e}'); return 0.5
#     try:
#         val_results = test_model(
#             model, full_returns, sectors, all_vol,
#             Market_caps, PE_ratios, Implied_vol, Short_interest,
#             Beta, Operating_margin, Return_on_equity, RSI_momentum,
#             Turnover, Z_DATA, emb_cache, val_dates, stock2idx,
#             K=K, k_neighbors=10, corr_threshold=0.0,
#         )
#         val_auc = compute_val_auc(val_results, val_prices)
#     except Exception as e:
#         print(f'    Eval failed: {e}'); val_auc = 0.5
#     if os.path.exists(save_path): os.remove(save_path)
#     print(f'    => Val AUC: {val_auc:.4f}')
#     return val_auc
#
# N_TRIALS_R2 = 30
# r2_study = optuna.create_study(
#     direction='maximize',
#     sampler=TPESampler(seed=0, n_startup_trials=15),
#     study_name='round2_architecture',
# )
# print(f'Starting Round 2 ({N_TRIALS_R2} trials)')
# r2_study.optimize(r2_objective, n_trials=N_TRIALS_R2)
# r2_best = r2_study.best_params
# print(f'Round 2 Best Val AUC: {r2_study.best_value:.4f}')
# for k, v in r2_best.items(): print(f'  {k} = {v}')


In [ ]:
# ================================================================
# ROUND 3 — Graph structure  (UNCOMMENT TO RUN)
# Paste Round 1 & 2 best params into R3_* variables, then uncomment.
# ================================================================

# R3_LR            = r1_best['lr']
# R3_WD            = r1_best['weight_decay']
# R3_K             = r1_best['K']
# R3_PATIENCE      = r1_best['patience']
# R3_EMBEDDING_DIM = r2_best['embedding_dim']
# R3_GAT_OUT       = r2_best['gat_output_dim']
# R3_GAT_HEADS     = r2_best['gat_heads']
# R3_HIDDEN_DIM    = r2_best['hidden_dim']
# R3_ENC_CH        = [r2_best['enc_ch1'], r2_best['enc_ch2']]
# R3_DIFF_STEPS    = r2_best['diffusion_steps']
# R3_DROPOUT       = r2_best['dropout']
# _r3_emb_cache    = {}

# def r3_objective(trial):
#     k_neighbors    = trial.suggest_int(  'k_neighbors',    3,  30)
#     corr_threshold = trial.suggest_float('corr_threshold', 0.0, 0.5)
#     print(f'  Trial {trial.number}: k_neighbors={k_neighbors}  '
#           f'corr_threshold={corr_threshold:.3f}')
#
#     K = R3_K
#     if K not in _r3_emb_cache:
#         _r3_emb_cache[K] = make_embedding_cache(K)
#     emb_cache = _r3_emb_cache[K]
#     train_dates, val_dates = make_date_ranges(K)
#
#     torch.manual_seed(SEED); np.random.seed(SEED)
#     gat_config = {'num_layers': 1, 'heads': [R3_GAT_HEADS],
#                   'features': [R3_EMBEDDING_DIM, R3_GAT_OUT]}
#     model = BubbleDetectionModel(
#         gat_config=gat_config, feature_dim=FEATURE_DIM,
#         embedding_dim=R3_EMBEDDING_DIM, encoder_channels=R3_ENC_CH,
#         hidden_dim=R3_HIDDEN_DIM, num_diffusion_steps=R3_DIFF_STEPS,
#         use_structure_recon=True, dropout=R3_DROPOUT,
#     )
#     optimizer = Adam(model.parameters(), lr=R3_LR, weight_decay=R3_WD)
#     save_path = f'outputs/bo_r3_t{trial.number}.pt'
#     os.makedirs('outputs', exist_ok=True)
#     try:
#         model, _, _ = train_model_bo(
#             model, optimizer, full_returns, sectors, all_vol,
#             Market_caps, PE_ratios, Implied_vol, Short_interest,
#             Beta, Operating_margin, Return_on_equity, RSI_momentum,
#             Turnover, Z_DATA, train_dates, stock2idx,
#             patience=R3_PATIENCE, val_dates=val_dates,
#             embedding_cache=emb_cache, K=K, save_path=save_path,
#             k_neighbors=k_neighbors, corr_threshold=corr_threshold,
#         )
#     except Exception as e:
#         print(f'    Training failed: {e}'); return 0.5
#     try:
#         val_results = test_model(
#             model, full_returns, sectors, all_vol,
#             Market_caps, PE_ratios, Implied_vol, Short_interest,
#             Beta, Operating_margin, Return_on_equity, RSI_momentum,
#             Turnover, Z_DATA, emb_cache, val_dates, stock2idx,
#             K=K, k_neighbors=k_neighbors, corr_threshold=corr_threshold,
#         )
#         val_auc = compute_val_auc(val_results, val_prices)
#     except Exception as e:
#         print(f'    Eval failed: {e}'); val_auc = 0.5
#     if os.path.exists(save_path): os.remove(save_path)
#     print(f'    => Val AUC: {val_auc:.4f}')
#     return val_auc
#
# N_TRIALS_R3 = 30
# r3_study = optuna.create_study(
#     direction='maximize',
#     sampler=TPESampler(seed=0, n_startup_trials=10),
#     study_name='round3_graph_structure',
# )
# print(f'Starting Round 3 ({N_TRIALS_R3} trials)')
# r3_study.optimize(r3_objective, n_trials=N_TRIALS_R3)
# r3_best = r3_study.best_params
# print(f'Round 3 Best Val AUC: {r3_study.best_value:.4f}')
# for k, v in r3_best.items(): print(f'  {k} = {v}')


In [ ]:
# ================================================================
# FINAL TRAINING — multi-seed retraining with all best params
# Update R2/R3 defaults below once those rounds complete.
# ================================================================

# Round 1 best (auto-populated after Round 1)
FINAL_LR      = r1_best['lr']
FINAL_WD      = r1_best['weight_decay']
FINAL_K       = r1_best['K']
FINAL_PATIENCE = r1_best['patience']

# Round 2 defaults — replace after Round 2 completes:
#   FINAL_EMBEDDING_DIM = r2_best['embedding_dim']
#   FINAL_GAT_OUT       = r2_best['gat_output_dim']
#   FINAL_GAT_HEADS     = r2_best['gat_heads']
#   FINAL_HIDDEN_DIM    = r2_best['hidden_dim']
#   FINAL_ENC_CH        = [r2_best['enc_ch1'], r2_best['enc_ch2']]
#   FINAL_DIFF_STEPS    = r2_best['diffusion_steps']
#   FINAL_DROPOUT       = r2_best['dropout']
FINAL_EMBEDDING_DIM = 10
FINAL_GAT_OUT       = 8
FINAL_GAT_HEADS     = 1
FINAL_HIDDEN_DIM    = 32
FINAL_ENC_CH        = [16, 8]
FINAL_DIFF_STEPS    = 1
FINAL_DROPOUT       = 0.1

# Round 3 defaults — replace after Round 3 completes:
#   FINAL_K_NEIGHBORS  = r3_best['k_neighbors']
#   FINAL_CORR_THR     = r3_best['corr_threshold']
FINAL_K_NEIGHBORS   = 10
FINAL_CORR_THR      = 0.0

K = FINAL_K
if K not in _r1_emb_cache:
    _r1_emb_cache[K] = make_embedding_cache(K)
emb_cache = _r1_emb_cache[K]

train_dates, val_dates = make_date_ranges(K)
test_min   = test_returns.index[0] + pd.Timedelta(days=K * 2)
test_dates = test_returns.loc[test_min:].index

gat_config = {
    'num_layers': 1,
    'heads':      [FINAL_GAT_HEADS],
    'features':   [FINAL_EMBEDDING_DIM, FINAL_GAT_OUT],
}
seeds = [0, 2, 4, 6]
all_seed_results = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    print(f'\n=== Seed {seed} ===')
    model = BubbleDetectionModel(
        gat_config=gat_config,
        feature_dim=FEATURE_DIM,
        embedding_dim=FINAL_EMBEDDING_DIM,
        encoder_channels=FINAL_ENC_CH,
        hidden_dim=FINAL_HIDDEN_DIM,
        num_diffusion_steps=FINAL_DIFF_STEPS,
        use_structure_recon=True,
        dropout=FINAL_DROPOUT,
    )
    optimizer = Adam(model.parameters(), lr=FINAL_LR, weight_decay=FINAL_WD)
    os.makedirs('outputs', exist_ok=True)

    model, train_hist, val_hist = train_model_bo(
        model, optimizer, full_returns, sectors, all_vol,
        Market_caps, PE_ratios, Implied_vol, Short_interest,
        Beta, Operating_margin, Return_on_equity, RSI_momentum,
        Turnover, Z_DATA, train_dates, stock2idx,
        patience=FINAL_PATIENCE, val_dates=val_dates,
        embedding_cache=emb_cache, K=K,
        save_path=f'outputs/bo_final_seed{seed}.pt',
        k_neighbors=FINAL_K_NEIGHBORS,
        corr_threshold=FINAL_CORR_THR,
    )

    plt.figure(figsize=(8, 3))
    plt.plot(train_hist, label='train loss')
    plt.plot(val_hist,   label='val loss')
    plt.legend(); plt.grid(alpha=0.3)
    plt.title(f'Final training — seed {seed}')
    plt.tight_layout(); plt.show()

    test_res = test_model(
        model, full_returns, sectors, all_vol,
        Market_caps, PE_ratios, Implied_vol, Short_interest,
        Beta, Operating_margin, Return_on_equity, RSI_momentum,
        Turnover, Z_DATA, emb_cache, test_dates, stock2idx,
        K=K, k_neighbors=FINAL_K_NEIGHBORS, corr_threshold=FINAL_CORR_THR,
    )
    all_seed_results.append(test_res)

# Average signals across seeds
final_results = {}
for res in all_seed_results:
    for date, (stocks, sigs) in res.items():
        if date not in final_results:
            final_results[date] = (stocks, [sigs])
        else:
            final_results[date][1].append(sigs)
for date in final_results:
    stocks, sigs_list = final_results[date]
    final_results[date] = (stocks, np.mean(sigs_list, axis=0))

# AUC grid on test set
print('\n=== Test set AUC grid ===')
for fw in [10, 22, 44]:
    for ct in [-0.10, -0.15, -0.20, -0.30]:
        r = evaluate_once(final_results, test_prices, fw, ct)
        if r:
            auc, lift, base = r
            print(f'FW={fw:3d}  CT={ct:5.2f}  AUC={auc:.3f}  Lift={lift:.2f}')

pd.to_pickle(final_results, 'outputs/test_results_bo_final.pkl')
pd.to_pickle(test_prices,   'outputs/test_prices_bo_final.pkl')
print('\nSaved to outputs/test_results_bo_final.pkl')
